# YOLO26 Tomato Plant Disease Classification Training Notebook

**Project Title**: Multi-View Drone-Based Agricultural Pest and Plant Disease Detection Using YOLO26 and Digital Image Processing for Plant Health Assessment

### Target Classes:
1. `healthy`
2. `early_blight`
3. `late_blight`
4. `septoria_leaf_spot`

Run this notebook in **Google Colab (T4 GPU)** to train the YOLO26 model on the PlantVillage dataset.

In [ ]:
# 1. Install required dependencies
!pip install -q ultralytics opencv-python matplotlib scikit-learn seaborn

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone or download PlantVillage dataset (Tomato 4 classes)
!git clone https://github.com/spMohanty/PlantVillage-Dataset.git /tmp/plantvillage

import os
import shutil
import random

DATASET_ROOT = "/tmp/plantvillage/raw/color"
TARGET_DIR = "/content/dataset"

CLASS_MAPPING = {
    "Tomato___healthy": "healthy",
    "Tomato___Early_blight": "early_blight",
    "Tomato___Late_blight": "late_blight",
    "Tomato___Septoria_leaf_spot": "septoria_leaf_spot"
}

# Create YOLO classification split layout
for split in ["train", "val", "test"]:
    for cls in CLASS_MAPPING.values():
        os.makedirs(os.path.join(TARGET_DIR, split, cls), exist_ok=True)

random.seed(42)
for orig_cls, target_cls in CLASS_MAPPING.items():
    src_folder = os.path.join(DATASET_ROOT, orig_cls)
    if not os.path.exists(src_folder):
        print(f"Warning: {src_folder} not found. Ensure dataset download.")
        continue
    images = [f for f in os.listdir(src_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)
    n_total = len(images)
    n_train = int(n_total * 0.70)
    n_val = int(n_total * 0.15)
    train_imgs = images[:n_train]
    val_imgs = images[n_train:n_train+n_val]
    test_imgs = images[n_train+n_val:]
    for img in train_imgs:
        shutil.copy(os.path.join(src_folder, img), os.path.join(TARGET_DIR, "train", target_cls, img))
    for img in val_imgs:
        shutil.copy(os.path.join(src_folder, img), os.path.join(TARGET_DIR, "val", target_cls, img))
    for img in test_imgs:
        shutil.copy(os.path.join(src_folder, img), os.path.join(TARGET_DIR, "test", target_cls, img))

print("Dataset partitioning complete: 70% Train, 15% Val, 15% Test.")

In [ ]:
# 3. Train YOLO26 Classification Model
from ultralytics import YOLO

# Load pretrained YOLO classification model
model = YOLO("yolov8n-cls.pt") # Ultralytics YOLO classification weights base

results = model.train(
    data="/content/dataset",
    epochs=15,
    imgsz=224,
    batch=32,
    name="yolo26_tomato_classifier"
)

In [ ]:
# 4. Validation and Confusion Matrix Evaluation
metrics = model.val()
print(f"Top-1 Accuracy: {metrics.top1:.4f}")
print(f"Top-5 Accuracy: {metrics.top5:.4f}")

In [ ]:
# 5. Export trained weights for backend placement
best_weights_path = os.path.join(model.trainer.save_dir, "weights", "best.pt")
output_target = "/content/yolo26_tomato.pt"
shutil.copy(best_weights_path, output_target)
print(f"Trained model weights saved to {output_target}")
print("DOWNLOAD INSTRUCTION: Download 'yolo26_tomato.pt' and place it in 'backend/models/' in your workspace.")